# 06 — LiDAR Range Finding

## Objective

Simulate a pulsed-laser rangefinder: estimate received power from the LiDAR range equation, compute round-trip time-of-flight, and turn the detected return into a point cloud.

## What you'll see

- Received-power estimate vs. range and backscatter (`LiDARRangeEquation`)
- Round-trip time-of-flight and pulse broadening (`TimeOfFlightPropagator`)
- Peak and timing extraction from a synthetic return (`WaveformAnalyzer`) and a Cartesian point cloud (`generate_point_cloud`)

The goal is to make the core LiDAR ranging workflow feel tangible and editable without requiring a full hardware simulation.

## How to use this notebook

Change the parameters in the next cell and rerun the later cells to see how the outputs change.

## How to use this notebook

- Edit the parameters in the next cell to change the range, backscatter, or pulse duration.
- Run the cells in order so you can see how each stage changes the result.
- Compare how a nearby target differs from a more distant one.
- Use the printed metrics as a guide to understand what changed.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
range_m = 12.0
backscatter = 1e-4
pulse_duration = 1e-9

print("Configuration:")
print(f"  range_m={range_m}")
print(f"  backscatter={backscatter}")
print(f"  pulse_duration={pulse_duration}")

In [ ]:
import numpy as np

from optical_metrology.analysis import LiDARRangeEquation, TimeOfFlightPropagator, WaveformAnalyzer, generate_point_cloud

In [ ]:
range_eq = LiDARRangeEquation(transmitter_power=1.0, receiver_aperture_diameter=0.1)
received_power = range_eq.compute_range(range_m, backscatter_coeff=backscatter)

print(f"Received power: {received_power:.6e}")

In [ ]:
tof_prop = TimeOfFlightPropagator()
tof, broadened_duration = tof_prop.compute_tof(range_m, pulse_duration=pulse_duration)

print(f"Time of flight: {tof:.6e}")
print(f"Broadened duration: {broadened_duration:.6e}")

In [ ]:
waveform = np.array([0.0, 0.2, 0.7, 1.0, 0.8, 0.4, 0.1], dtype=float)
analysis = WaveformAnalyzer(cf_fraction=0.5).analyze(waveform)

print("Waveform summary:")
for key, value in analysis.measurements.items():
    print(f"  {key}: {value}")

In [ ]:
azimuths = np.array([0.0, 0.4, 0.8])
elevations = np.array([0.0, 0.1, -0.1])
ranges = np.array([range_m, range_m + 1.0, range_m + 2.0])
point_cloud = generate_point_cloud(ranges, azimuths, elevations)

print("Point cloud shape:")
print(point_cloud.shape)
print(point_cloud)

## Try next

Try changing one thing at a time:
- reduce or increase the range to see how the received power changes
- change the backscatter coefficient to model a brighter or darker return
- increase the pulse duration to see the effect on timing broadening
- add a tilt to the target to see how the pulse broadened duration changes